# End-to-End FEVER/KILT Fact-Checking Demo

This notebook runs the project pipeline in one place:

1. Download FEVER claim data.
2. Prepare a KILT passage corpus with preprocessing matched to the retrieval method.
3. Preprocess FEVER claims with the same text variant used for the KILT corpus.
4. Build retrieval embeddings/indexes with `tfidf`, `word2vec`, or `transformer`.
5. Retrieve evidence for one claim.
6. Call an LLM with the claim plus retrieved evidence and predict whether the claim is true or fake/false.

Preprocessing rule:

- `transformer` uses the minimal/readable KILT and FEVER text.
- `tfidf` and `word2vec` use the processed KILT and FEVER text.

Default tradeoff: downloading the prebuilt KILT/FEVER ZIPs from Box is much faster than downloading and processing the full KILT knowledge source. Set `KILT_CORPUS_SOURCE = "full_kilt_source"` only when you want to download and process the full official KILT knowledge source.


In [33]:
# =========================
# Configuration constants
# =========================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "end_to_end_demo"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# FEVER data download. The labelled dev split is small enough for a demo.
FEVER_URLS = {
    "train": "https://fever.ai/download/fever/train.jsonl",
    "labelled_dev": "https://fever.ai/download/fever/shared_task_dev.jsonl",
}
FEVER_SPLIT = "labelled_dev"
FEVER_JSONL_PATH = DATA_DIR / f"fever_{FEVER_SPLIT}.jsonl"

# Retrieval method. Set RUN_ALL_METHODS=True to build and compare all three.
RETRIEVAL_METHOD = "word2vec"  # choices: "tfidf", "word2vec", "transformer"
RUN_ALL_METHODS = False

# Enforced preprocessing rule. Do not edit unless your project definition changes.
METHOD_TEXT_VARIANT = {
    "tfidf": "processed",
    "word2vec": "processed",
    "transformer": "minimal",
}

# KILT corpus source. Default is the fast Box download.
# Choices:
# - "box_zip": download the prebuilt ZIP artifact that matches each retrieval method.
# - "full_kilt_source": download official kilt_knowledgesource.json and process every KILT page.
# - "local_kilt_jsonl": use already-extracted local folders of KILT-style passage JSONL files.
KILT_CORPUS_SOURCE = "box_zip"

BOX_DOWNLOAD_DIR = DATA_DIR / "box_download"
BOX_ARTIFACT_FILENAMES = {
    "minimal": "kilt_passages_minimal_fever_evidence.zip",
    "processed": "kilt_passages_processed_fever_evidence_merged.zip",
}
BOX_DIRECT_ZIP_URLS = {
    "minimal": "https://uofi.box.com/s/rs28q8gwby7pfql4v1m7d7ze4xu9sifg",
    "processed": "https://uofi.box.com/s/x30jokwnpwh81abic2muqtdckmdw26u0",
}

# Full-source KILT option. This is about 37GB before preprocessing and writes many passage shards.
KILT_SOURCE_URL = "http://dl.fbaipublicfiles.com/KILT/kilt_knowledgesource.json"
KILT_SOURCE_JSON_PATH = PROJECT_ROOT.parent / "kilt_knowledgesource.json"
FULL_KILT_PASSAGE_DIRS = {
    "minimal": DATA_DIR / "full_kilt_passages_minimal",
    "processed": DATA_DIR / "full_kilt_passages_processed",
}
KILT_SOURCE_SHARD_SIZE = 500_000

# Existing local KILT-style JSONL options.
LOCAL_KILT_PASSAGE_DIRS = {
    "minimal": PROJECT_ROOT / "data" / "kilt" / "kilt_passages_minimal_fever_evidence",
    "processed": PROJECT_ROOT / "data" / "kilt" / "kilt_passages_processed_fever_evidence_merged",
}

# Corpus loading controls. None means use all available records from the selected corpus.
MAX_CORPUS_DOCS = None
MAX_CORPUS_PASSAGES = None
MAX_PASSAGES_PER_DOC = None
MIN_PASSAGE_CHARS = 40

# Demo claim controls.
RANDOM_SEED = 42
MAX_FEVER_ROWS_TO_SCAN = 2000
MAX_CLAIMS = 40
DEMO_CLAIM_INDEX = 20
TOP_DOCS = 5
TOP_PASSAGE_CANDIDATES = 30

# TF-IDF settings.
TFIDF_MAX_FEATURES = 100_000
TFIDF_MIN_DF = 1
TFIDF_NGRAM_RANGE = (1, 2)

# Word2Vec settings. The default trains a Word2Vec model on the selected processed KILT corpus.
# For the larger pretrained setup, set WORD2VEC_USE_PRETRAINED=True;
# note that word2vec-google-news-300 is a large download.
WORD2VEC_USE_PRETRAINED = False
WORD2VEC_PRETRAINED_NAME = "word2vec-google-news-300"
WORD2VEC_VECTOR_SIZE = 100
WORD2VEC_WINDOW = 5
WORD2VEC_EPOCHS = 20

# Transformer retrieval settings.
TRANSFORMER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TRANSFORMER_BATCH_SIZE = 64

# LLM settings. google/flan-t5-base is small enough for a demo; swap to your class model if needed.
RUN_LLM = True
LLM_TASK = "text2text-generation"  # "text2text-generation" for FLAN-T5; "text-generation" for causal LMs
LLM_MODEL_ID = "google/flan-t5-base"
LLM_MAX_INPUT_CHARS = 5000
LLM_MAX_NEW_TOKENS = 12

# Dependency installation. Set False if your environment is already prepared.
INSTALL_MISSING_PACKAGES = True


In [34]:
# =========================
# Optional dependency install
# =========================
import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    "requests": "requests",
    "tqdm": "tqdm",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "gensim": "gensim",
    "nltk": "nltk",
    "sentence-transformers": "sentence_transformers",
    "transformers": "transformers",
    "torch": "torch",
}

if INSTALL_MISSING_PACKAGES:
    missing = [pkg for pkg, mod in PACKAGE_IMPORTS.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("All required packages are already importable.")


All required packages are already importable.


In [35]:
# =========================
# Imports and helpers
# =========================
import json
import random
import re
import time
import unicodedata
import zipfile
from urllib.parse import unquote

import numpy as np
import requests
from tqdm.auto import tqdm

try:
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    import nltk
    try:
        STOP_WORDS = set(stopwords.words("english"))
    except LookupError:
        nltk.download("stopwords")
        STOP_WORDS = set(stopwords.words("english"))
    STEMMER = PorterStemmer()
except Exception as exc:
    print(f"NLTK stopword/stemmer setup failed ({exc}); using a small fallback stopword list.")
    STOP_WORDS = {
        "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "he", "in", "is", "it",
        "its", "of", "on", "that", "the", "to", "was", "were", "will", "with",
    }
    STEMMER = None

TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_'-]*")


def download_file(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        print(f"Using existing file: {path}")
        return path

    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {path}")
    with requests.get(url, stream=True, timeout=60, allow_redirects=True) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with path.open("wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=path.name) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    # Box sometimes returns an HTML page when a shared link is not directly downloadable.
    with path.open("rb") as f:
        prefix = f.read(256).lower()
    if prefix.lstrip().startswith(b"<!doctype html") or prefix.lstrip().startswith(b"<html"):
        raise RuntimeError(f"Downloaded HTML instead of a data file from {url}.")
    return path


def iter_jsonl(path, limit=None):
    with Path(path).open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if line:
                yield json.loads(line)


def write_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(rows):,} rows to {path}")


def clean_text(text, lowercase=False):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower() if lowercase else text


def minimal_text(text):
    text = clean_text(text)
    if text.startswith("Section::::"):
        return ""
    if text.startswith("BULLET::::"):
        text = text[len("BULLET::::"):].strip()
    return clean_text(text)


def processed_text(text):
    text = minimal_text(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = []
    for word in text.split():
        if word in STOP_WORDS:
            continue
        tokens.append(STEMMER.stem(word) if STEMMER is not None else word)
    return " ".join(tokens)


def preprocess_text(text, variant):
    if variant == "minimal":
        return minimal_text(text)
    if variant == "processed":
        return processed_text(text)
    raise ValueError(f"Unknown text variant: {variant}")


def tokenize(text):
    return [m.group(0).lower() for m in TOKEN_RE.finditer(text or "")]


def fever_title_to_wikipedia_title(title):
    title = str(title or "")
    replacements = {
        "-LRB-": "(",
        "-RRB-": ")",
        "-LSB-": "[",
        "-RSB-": "]",
        "-LCB-": "{",
        "-RCB-": "}",
        "-COLON-": ":",
    }
    for old, new in replacements.items():
        title = title.replace(old, new)
    return unquote(title.replace("_", " ")).strip()


def doc_id_from_title(title):
    title = fever_title_to_wikipedia_title(title)
    return re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_") or "doc"


def evidence_titles(row):
    titles = []
    for group in row.get("evidence") or []:
        for item in group or []:
            if len(item) >= 3 and item[2]:
                titles.append(fever_title_to_wikipedia_title(item[2]))
    return sorted(set(titles))


def normalize_title(title):
    return fever_title_to_wikipedia_title(title).lower().strip()


def required_text_variants(methods):
    variants = []
    for method in methods:
        variant = METHOD_TEXT_VARIANT[method]
        if variant not in variants:
            variants.append(variant)
    return variants


In [36]:
# =========================
# Download and preprocess FEVER claims
# =========================
random.seed(RANDOM_SEED)

if FEVER_SPLIT not in FEVER_URLS:
    raise ValueError(f"Unknown FEVER_SPLIT={FEVER_SPLIT!r}; choose one of {sorted(FEVER_URLS)}")

available_methods = sorted(METHOD_TEXT_VARIANT)
methods_to_run = available_methods if RUN_ALL_METHODS else [RETRIEVAL_METHOD]
unknown = set(methods_to_run) - set(METHOD_TEXT_VARIANT)
if unknown:
    raise ValueError(f"Unknown retrieval method(s): {sorted(unknown)}")

text_variants_to_prepare = required_text_variants(methods_to_run)
print("Methods to run:", methods_to_run)
print("Required text variants:", text_variants_to_prepare)

download_file(FEVER_URLS[FEVER_SPLIT], FEVER_JSONL_PATH)

raw_rows = list(iter_jsonl(FEVER_JSONL_PATH, limit=MAX_FEVER_ROWS_TO_SCAN))
usable_rows = [
    row for row in raw_rows
    if row.get("label") in {"SUPPORTS", "REFUTES"} and evidence_titles(row)
]

if len(usable_rows) < MAX_CLAIMS:
    print(f"Only found {len(usable_rows)} usable SUPPORTS/REFUTES rows with evidence.")

raw_claims = usable_rows[:MAX_CLAIMS]
claims_by_variant = {}
for variant in text_variants_to_prepare:
    variant_claims = []
    for row in raw_claims:
        out = dict(row)
        out["raw_claim"] = clean_text(row.get("claim"))
        out["claim"] = preprocess_text(row.get("claim"), variant)
        out["text_variant"] = variant
        variant_claims.append(out)
    claims_by_variant[variant] = variant_claims
    write_jsonl(variant_claims, DATA_DIR / f"claims_sample_{variant}.jsonl")

# Keep a default claims variable for quick inspection. Retrieval uses claims_by_variant.
default_variant = METHOD_TEXT_VARIANT[RETRIEVAL_METHOD]
claims = claims_by_variant[default_variant]

print(f"Loaded {len(raw_claims):,} demo claims from {FEVER_JSONL_PATH}")
print("Label counts:", {label: sum(row["label"] == label for row in raw_claims) for label in ["SUPPORTS", "REFUTES"]})
for variant in text_variants_to_prepare:
    print(f"\nDemo claim ({variant}):", claims_by_variant[variant][DEMO_CLAIM_INDEX]["claim"])
print("Gold label:", raw_claims[DEMO_CLAIM_INDEX]["label"])
print("Gold evidence pages:", evidence_titles(raw_claims[DEMO_CLAIM_INDEX])[:5])


Methods to run: ['word2vec']
Required text variants: ['processed']
Using existing file: c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\fever_labelled_dev.jsonl
Wrote 40 rows to c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\claims_sample_processed.jsonl
Loaded 40 demo claims from c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\fever_labelled_dev.jsonl
Label counts: {'SUPPORTS': 15, 'REFUTES': 25}

Demo claim (processed): hot right mistakenli attribut dj fresh
Gold label: REFUTES
Gold evidence pages: ['Hot Right Now']


In [37]:
# =========================
# Prepare KILT passage corpus variants
# =========================
def extract_zip(zip_path, output_dir):
    output_dir = Path(output_dir)
    marker = output_dir / ".extracted"
    if marker.exists() and list(output_dir.rglob("*.jsonl")):
        print(f"Using existing extracted corpus: {output_dir}")
        return output_dir

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path} -> {output_dir}")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(output_dir)
    marker.write_text("ok", encoding="utf-8")
    return output_dir


def normalize_box_download_url(url):
    # A Box file shared link like https://uofi.box.com/s/<id> can usually be
    # downloaded with the equivalent /shared/static/<id> URL.
    if url and "box.com/s/" in url:
        return url.replace("/s/", "/shared/static/")
    return url


def download_box_artifact(variant):
    if variant not in BOX_ARTIFACT_FILENAMES:
        raise ValueError(f"No Box artifact configured for text variant {variant!r}")

    BOX_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    artifact_filename = BOX_ARTIFACT_FILENAMES[variant]
    artifact_zip_path = BOX_DOWNLOAD_DIR / artifact_filename
    direct_url = BOX_DIRECT_ZIP_URLS.get(variant)
    if not direct_url:
        raise ValueError(f"BOX_DIRECT_ZIP_URLS[{variant!r}] is not set")

    download_file(normalize_box_download_url(direct_url), artifact_zip_path)
    extract_dir = DATA_DIR / "kilt_box_corpus" / variant
    return extract_zip(artifact_zip_path, extract_dir)


def process_full_kilt_source(source_json_path, output_dir, variant):
    output_dir = Path(output_dir)
    marker = output_dir / f".processed_full_kilt_{variant}"
    if marker.exists() and list(output_dir.glob("kilt_passages_*.jsonl")):
        print(f"Using existing full KILT {variant} passage shards: {output_dir}")
        return output_dir

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Processing the full KILT knowledge source as {variant} text. This can take a long time.")

    fout = None
    shard_idx = 0
    passages_in_shard = 0
    total_passages = 0
    total_docs = 0

    try:
        with Path(source_json_path).open("r", encoding="utf-8") as fin:
            for line in tqdm(fin, desc=f"Processing KILT pages ({variant})"):
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                doc_id = str(entry.get("wikipedia_id") or doc_id_from_title(entry.get("wikipedia_title")))
                title = clean_text(entry.get("wikipedia_title"))
                raw_passages = entry.get("text") or []

                for idx, raw_passage in enumerate(raw_passages):
                    text = preprocess_text(raw_passage, variant)
                    if len(text) < MIN_PASSAGE_CHARS:
                        continue

                    if fout is None or passages_in_shard >= KILT_SOURCE_SHARD_SIZE:
                        if fout is not None:
                            fout.close()
                        shard_path = output_dir / f"kilt_passages_{shard_idx:05d}.jsonl"
                        fout = shard_path.open("w", encoding="utf-8")
                        shard_idx += 1
                        passages_in_shard = 0

                    row = {
                        "passage_id": f"{doc_id}_{idx}",
                        "doc_id": doc_id,
                        "title": title,
                        "text": text,
                    }
                    fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                    passages_in_shard += 1
                    total_passages += 1

                total_docs += 1
                if total_docs % 10000 == 0:
                    print(f"Processed {total_docs:,} KILT pages and wrote {total_passages:,} {variant} passages")
    finally:
        if fout is not None:
            fout.close()

    marker.write_text(json.dumps({"variant": variant, "docs": total_docs, "passages": total_passages}), encoding="utf-8")
    print(f"Finished full KILT {variant} processing: {total_docs:,} docs, {total_passages:,} passages")
    return output_dir


def prepare_kilt_corpus(variant):
    if KILT_CORPUS_SOURCE == "box_zip":
        return download_box_artifact(variant)
    if KILT_CORPUS_SOURCE == "full_kilt_source":
        download_file(KILT_SOURCE_URL, KILT_SOURCE_JSON_PATH)
        return process_full_kilt_source(KILT_SOURCE_JSON_PATH, FULL_KILT_PASSAGE_DIRS[variant], variant)
    if KILT_CORPUS_SOURCE == "local_kilt_jsonl":
        corpus_dir = LOCAL_KILT_PASSAGE_DIRS[variant]
        if not Path(corpus_dir).exists():
            raise FileNotFoundError(f"LOCAL_KILT_PASSAGE_DIRS[{variant!r}] does not exist: {corpus_dir}")
        return Path(corpus_dir)
    raise ValueError("KILT_CORPUS_SOURCE must be 'box_zip', 'full_kilt_source', or 'local_kilt_jsonl'")


def locate_jsonl_files(corpus_dir):
    files = sorted(Path(corpus_dir).rglob("*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No .jsonl files found under {corpus_dir}")
    return files


def load_kilt_docs(corpus_dir):
    docs_by_id = {}
    total_passages = 0

    for path in locate_jsonl_files(corpus_dir):
        for row in iter_jsonl(path):
            doc_id = str(row.get("doc_id"))
            title = clean_text(row.get("title"))
            text = clean_text(row.get("text"))
            if len(text) < MIN_PASSAGE_CHARS:
                continue

            if doc_id not in docs_by_id:
                if MAX_CORPUS_DOCS is not None and len(docs_by_id) >= MAX_CORPUS_DOCS:
                    return list(docs_by_id.values())
                docs_by_id[doc_id] = {"doc_id": doc_id, "title": title, "passages": []}

            if MAX_PASSAGES_PER_DOC is None or len(docs_by_id[doc_id]["passages"]) < MAX_PASSAGES_PER_DOC:
                docs_by_id[doc_id]["passages"].append({
                    "passage_id": str(row.get("passage_id")),
                    "text": text,
                })
                total_passages += 1

            if MAX_CORPUS_PASSAGES is not None and total_passages >= MAX_CORPUS_PASSAGES:
                return list(docs_by_id.values())

    return list(docs_by_id.values())


def flatten_passages(docs):
    flat = []
    for doc in docs:
        for passage in doc["passages"]:
            flat.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "passage_id": passage["passage_id"],
                "text": passage["text"],
                "retrieval_text": clean_text(f"{doc['title']} {passage['text']}", lowercase=True),
            })
    return flat


corpora_by_variant = {}
for variant in text_variants_to_prepare:
    corpus_dir = prepare_kilt_corpus(variant)
    variant_docs = load_kilt_docs(corpus_dir)
    if not variant_docs:
        raise RuntimeError(f"The {variant} KILT corpus is empty. Check KILT_CORPUS_SOURCE and corpus paths.")

    variant_passages = flatten_passages(variant_docs)
    corpora_by_variant[variant] = {
        "corpus_dir": corpus_dir,
        "docs": variant_docs,
        "passages": variant_passages,
        "doc_index": {doc["doc_id"]: doc for doc in variant_docs},
    }
    write_jsonl(variant_docs, DATA_DIR / f"corpus_docs_{variant}.jsonl")
    write_jsonl(variant_passages, DATA_DIR / f"corpus_passages_{variant}.jsonl")

    print(f"\nCorpus variant: {variant}")
    print(f"Corpus source: {KILT_CORPUS_SOURCE}")
    print(f"Corpus directory: {corpus_dir}")
    print(f"Corpus docs: {len(variant_docs):,}")
    print(f"Corpus passages: {len(variant_passages):,}")
    print("Sample doc:", variant_docs[0]["title"])
    print("Sample passage:", variant_docs[0]["passages"][0]["text"][:300])

# Keep default globals for quick inspection. Retrieval switches these per method.
docs = corpora_by_variant[default_variant]["docs"]
passages = corpora_by_variant[default_variant]["passages"]
doc_index = corpora_by_variant[default_variant]["doc_index"]


Using existing file: c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\box_download\kilt_passages_processed_fever_evidence_merged.zip
Using existing extracted corpus: c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\kilt_box_corpus\processed
Wrote 13,864 rows to c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\corpus_docs_processed.jsonl
Wrote 589,664 rows to c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\corpus_passages_processed.jsonl

Corpus variant: processed
Corpus source: box_zip
Corpus directory: c:\Users\miche\Documents\UIUC\spring2026\cs410\project\CS410_project\data\end_to_end_demo\kilt_box_corpus\processed
Corpus docs: 13,864
Corpus passages: 589,664
Sample doc: Academy Awards
Sample passage: academi award also offici popularli known oscar award artist technic merit film industri given annual academi motion pictu

In [28]:
# =========================
# Shared retrieval utilities
# =========================
def dedupe_passage_hits(hit_indices, hit_scores, top_docs=TOP_DOCS):
    seen = set()
    retrieved = []
    for idx, score in zip(hit_indices, hit_scores):
        passage = passages[int(idx)]
        doc_id = passage["doc_id"]
        if doc_id in seen:
            continue
        seen.add(doc_id)
        retrieved.append({
            "doc_id": doc_id,
            "title": passage["title"],
            "score": round(float(score), 6),
            "best_passage_id": passage["passage_id"],
            "text": passage["text"],
        })
        if len(retrieved) >= top_docs:
            break
    return retrieved


def print_retrieval_result(method, claim_row, retrieved):
    print(f"\n=== {method.upper()} retrieval ===")
    print("Claim:", claim_row["claim"])
    print("Gold label:", claim_row["label"])
    print("Gold evidence pages:", evidence_titles(claim_row))
    for rank, item in enumerate(retrieved, start=1):
        print(f"{rank}. {item['title']}  score={item['score']}  doc_id={item['doc_id']}")
        print("   ", item["text"][:220].replace("\n", " "))


def gold_title_hit(claim_row, retrieved):
    gold = {normalize_title(t) for t in evidence_titles(claim_row)}
    got = {normalize_title(item["title"]) for item in retrieved}
    return bool(gold & got)


In [29]:
# =========================
# Retrieval method: TF-IDF
# =========================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


def build_tfidf_index():
    print(f"Preparing {len(passages):,} passages for TF-IDF...")
    passage_texts = [p["retrieval_text"] for p in tqdm(passages, desc="Collecting TF-IDF texts")]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        dtype=np.float32,
    )
    print("Fitting TF-IDF vectorizer and building sparse passage matrix...")
    start = time.time()
    matrix = vectorizer.fit_transform(passage_texts)
    print(f"TF-IDF matrix shape: {matrix.shape}; vocab size: {len(vectorizer.vocabulary_):,}; time: {time.time() - start:.1f}s")
    return {"vectorizer": vectorizer, "matrix": matrix}


def retrieve_tfidf(index_data, claim, top_docs=TOP_DOCS):
    print("Scoring claim against TF-IDF passage matrix...")
    query = index_data["vectorizer"].transform([clean_text(claim, lowercase=True)])
    scores = linear_kernel(query, index_data["matrix"]).ravel()
    n = min(TOP_PASSAGE_CANDIDATES, len(scores))
    top_idx = np.argpartition(-scores, n - 1)[:n]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    return dedupe_passage_hits(top_idx, scores[top_idx], top_docs=top_docs)


In [ ]:
# =========================
# Retrieval method: Word2Vec
# =========================
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
import gensim.downloader as gensim_api


def average_vectors(tokens, keyed_vectors, vector_size):
    vectors = []
    for token in tokens:
        if token in keyed_vectors:
            vectors.append(np.asarray(keyed_vectors[token], dtype=np.float32))
    if not vectors:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)


def normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return matrix / norms


class Word2VecEpochLogger(CallbackAny2Vec):
    def __init__(self, total_epochs):
        self.epoch = 0
        self.total_epochs = total_epochs
        self.start_time = None

    def on_train_begin(self, model):
        self.start_time = time.time()
        print("Word2Vec training started")

    def on_epoch_end(self, model):
        self.epoch += 1
        elapsed = time.time() - self.start_time
        print(f"  epoch {self.epoch}/{self.total_epochs} complete ({elapsed:.1f}s elapsed)", flush=True)


def build_word2vec_index():
    print(f"Tokenizing {len(passages):,} passages for Word2Vec...")
    tokenized_passages = [
        tokenize(p["retrieval_text"])
        for p in tqdm(passages, desc="Tokenizing passages")
    ]
    tokenized_claims = [
        tokenize(row["claim"])
        for row in tqdm(claims, desc="Tokenizing claims")
    ]

    if WORD2VEC_USE_PRETRAINED:
        print(f"Loading pretrained Word2Vec vectors: {WORD2VEC_PRETRAINED_NAME}")
        keyed_vectors = gensim_api.load(WORD2VEC_PRETRAINED_NAME)
        vector_size = keyed_vectors.vector_size
        print(f"Loaded pretrained vectors: {len(keyed_vectors):,} tokens, dim={vector_size}")
    else:
        sentences = tokenized_passages + tokenized_claims
        print(f"Training Word2Vec on {len(sentences):,} tokenized texts for {WORD2VEC_EPOCHS} epochs...")
        start = time.time()
        model = Word2Vec(
            sentences=sentences,
            vector_size=WORD2VEC_VECTOR_SIZE,
            window=WORD2VEC_WINDOW,
            min_count=1,
            workers=4,
            seed=RANDOM_SEED,
            epochs=WORD2VEC_EPOCHS,
            callbacks=[Word2VecEpochLogger(WORD2VEC_EPOCHS)],
        )
        keyed_vectors = model.wv
        vector_size = model.vector_size
        print(f"Word2Vec training complete in {time.time() - start:.1f}s; vocab size: {len(keyed_vectors):,}")

    print(f"Embedding {len(tokenized_passages):,} passages with Word2Vec averages...")
    matrix = np.vstack([
        average_vectors(tokens, keyed_vectors, vector_size)
        for tokens in tqdm(tokenized_passages, desc="Embedding passages")
    ]).astype(np.float32)
    matrix = normalize_rows(matrix)
    print(f"Word2Vec passage matrix shape: {matrix.shape}")
    return {"keyed_vectors": keyed_vectors, "vector_size": vector_size, "matrix": matrix}


def retrieve_word2vec(index_data, claim, top_docs=TOP_DOCS):
    print("Scoring claim against Word2Vec passage matrix...")
    query = average_vectors(tokenize(claim), index_data["keyed_vectors"], index_data["vector_size"])
    query = query.reshape(1, -1)
    query = normalize_rows(query)
    scores = (query @ index_data["matrix"].T).ravel()
    n = min(TOP_PASSAGE_CANDIDATES, len(scores))
    top_idx = np.argpartition(-scores, n - 1)[:n]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    return dedupe_passage_hits(top_idx, scores[top_idx], top_docs=top_docs)


In [31]:
# =========================
# Retrieval method: Transformer
# =========================
from sentence_transformers import SentenceTransformer


def build_transformer_index():
    print(f"Loading transformer retrieval model: {TRANSFORMER_MODEL_NAME}")
    model = SentenceTransformer(TRANSFORMER_MODEL_NAME)
    passage_texts = [p["retrieval_text"] for p in tqdm(passages, desc="Collecting transformer texts")]
    print(f"Encoding {len(passage_texts):,} passages with batch size {TRANSFORMER_BATCH_SIZE}...")
    start = time.time()
    matrix = model.encode(
        passage_texts,
        batch_size=TRANSFORMER_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Transformer passage matrix shape: {matrix.shape}; time: {time.time() - start:.1f}s")
    return {"model": model, "matrix": matrix}


def retrieve_transformer(index_data, claim, top_docs=TOP_DOCS):
    print("Encoding and scoring claim with transformer index...")
    query = index_data["model"].encode(
        [claim],
        batch_size=1,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    scores = (query @ index_data["matrix"].T).ravel()
    n = min(TOP_PASSAGE_CANDIDATES, len(scores))
    top_idx = np.argpartition(-scores, n - 1)[:n]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    return dedupe_passage_hits(top_idx, scores[top_idx], top_docs=top_docs)


In [ ]:
# =========================
# Build retrieval index(es) and retrieve for one claim
# =========================
BUILDERS = {
    "tfidf": build_tfidf_index,
    "word2vec": build_word2vec_index,
    "transformer": build_transformer_index,
}
RETRIEVERS = {
    "tfidf": retrieve_tfidf,
    "word2vec": retrieve_word2vec,
    "transformer": retrieve_transformer,
}

retrieval_outputs = {}
retrieval_indexes = {}
claim_rows_by_method = {}

for method in methods_to_run:
    text_variant = METHOD_TEXT_VARIANT[method]
    print(f"\nBuilding {method} index with {text_variant} FEVER/KILT text...")

    # The retrieval helpers use these globals, so set them to the method-matched variant.
    claims = claims_by_variant[text_variant]
    docs = corpora_by_variant[text_variant]["docs"]
    passages = corpora_by_variant[text_variant]["passages"]
    doc_index = corpora_by_variant[text_variant]["doc_index"]
    claim_row = claims[DEMO_CLAIM_INDEX]
    claim_rows_by_method[method] = claim_row

    print(f"Index input: {len(docs):,} docs, {len(passages):,} passages, {len(claims):,} claims")
    start = time.time()
    retrieval_indexes[method] = BUILDERS[method]()
    print(f"Retrieving top {TOP_DOCS} docs for demo claim with {method}...")
    retrieved = RETRIEVERS[method](retrieval_indexes[method], claim_row["claim"], top_docs=TOP_DOCS)
    retrieval_outputs[method] = retrieved
    print(f"Built and searched {method} in {time.time() - start:.1f}s")
    print_retrieval_result(method, claim_row, retrieved)
    print("Gold-title hit:", gold_title_hit(claim_row, retrieved))

selected_variant = METHOD_TEXT_VARIANT[RETRIEVAL_METHOD]
claim_row = claim_rows_by_method[RETRIEVAL_METHOD]
selected_retrieved = retrieval_outputs[RETRIEVAL_METHOD]
docs = corpora_by_variant[selected_variant]["docs"]
passages = corpora_by_variant[selected_variant]["passages"]
doc_index = corpora_by_variant[selected_variant]["doc_index"]
print(f"\nSelected method for LLM prompt: {RETRIEVAL_METHOD} ({selected_variant} text)")



Building word2vec index with processed FEVER/KILT text...
Index input: 13,864 docs, 589,664 passages, 40 claims
Tokenizing 589,664 passages for Word2Vec...


Tokenizing claims: 100%|██████████| 40/40 [00:00<?, ?it/s]


Training Word2Vec on 589,704 tokenized texts for 20 epochs...


In [ ]:
# =========================
# Build the LLM prompt from retrieved evidence
# =========================
def build_evidence_context(retrieved, max_chars=LLM_MAX_INPUT_CHARS):
    parts = []
    for i, item in enumerate(retrieved, start=1):
        parts.append(f"[{i}] Title: {item['title']}\nPassage: {item['text']}")
    context = "\n\n".join(parts)
    if len(context) > max_chars:
        context = context[:max_chars]
    return context


def build_llm_prompt(claim, retrieved):
    evidence = build_evidence_context(retrieved)
    return f"""You are a careful fact checker.
Use the evidence passages to decide whether the claim is true or false.
Return exactly one label: SUPPORTS or REFUTES.
SUPPORTS means the claim is true / not fake.
REFUTES means the claim is false / fake.

Evidence:
{evidence}

Claim: {claim}

Label:"""


def parse_llm_label(text):
    cleaned = (text or "").upper()
    if "REFUTES" in cleaned or "REFUTE" in cleaned or "FALSE" in cleaned or "FAKE" in cleaned:
        return "REFUTES"
    if "SUPPORTS" in cleaned or "SUPPORT" in cleaned or "TRUE" in cleaned:
        return "SUPPORTS"
    return "UNPARSEABLE"

prompt = build_llm_prompt(claim_row["claim"], selected_retrieved)
print(prompt[:2500])


You are a careful fact checker.
Use the evidence passages to decide whether the claim is true or false.
Return exactly one label: SUPPORTS or REFUTES.
SUPPORTS means the claim is true / not fake.
REFUTES means the claim is false / fake.

Evidence:
[1] Title: Hot Right Now
Passage: juli dj fresh releas louder first singl third studio album nextlevel reach peak posit uk singl chart give dj fresh first ever number one louder fresh start work hot right look vocalist song came across ora cover video youtub daniel stein invis men wrote song daniel stein wez clark produc

[2] Title: Disc jockey
Passage: club dj commonli refer dj gener play music music event parti music venu bar music festiv corpor privat event typic club dj mix music record two sourc use differ mix techniqu order produc non stop flow music one key techniqu use seamlessli transit one song anoth beatmatch dj mostli play mix one specif music genr often given titl genr exampl dj play hip hop music call hip hop dj dj play hous mus

In [ ]:
# =========================
# Call the LLM for a final true/fake verdict
# =========================
if RUN_LLM:
    import torch
    from transformers import pipeline

    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading LLM: {LLM_MODEL_ID} on device={device}")
    generator = pipeline(LLM_TASK, model=LLM_MODEL_ID, device=device)

    if LLM_TASK == "text2text-generation":
        output = generator(prompt, max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=False)[0]["generated_text"]
    elif LLM_TASK == "text-generation":
        generated = generator(prompt, max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=False)[0]["generated_text"]
        output = generated[len(prompt):].strip() if generated.startswith(prompt) else generated
    else:
        raise ValueError("LLM_TASK must be 'text2text-generation' or 'text-generation'")

    predicted_label = parse_llm_label(output)
    fake_status = "FAKE / false" if predicted_label == "REFUTES" else "NOT FAKE / true" if predicted_label == "SUPPORTS" else "UNPARSEABLE"

    print("Raw LLM output:", repr(output))
    print("Parsed label:", predicted_label)
    print("Fake/not-fake prediction:", fake_status)
    print("Gold FEVER label:", claim_row["label"])
else:
    print("RUN_LLM=False, so the notebook stops after building the prompt.")


Loading LLM: google/flan-t5-base on device=-1


Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1170 > 512). Running this sequence through the model will result in indexing errors


Raw LLM output: 'REFUTES'
Parsed label: REFUTES
Fake/not-fake prediction: FAKE / false
Gold FEVER label: REFUTES


In [ ]:
# =========================
# Optional: save one demo record for your report
# =========================
demo_record = {
    "claim_id": claim_row.get("id"),
    "claim": claim_row["claim"],
    "raw_claim": claim_row.get("raw_claim"),
    "gold_label": claim_row["label"],
    "retrieval_method": RETRIEVAL_METHOD,
    "text_variant": selected_variant,
    "retrieved": selected_retrieved,
}

if "predicted_label" in globals():
    demo_record["llm_raw_output"] = output
    demo_record["predicted_label"] = predicted_label

DEMO_OUTPUT_PATH = DATA_DIR / f"single_claim_demo_{RETRIEVAL_METHOD}_{selected_variant}.json"
DEMO_OUTPUT_PATH.write_text(json.dumps(demo_record, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved demo record to {DEMO_OUTPUT_PATH}")
